In [38]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import os

from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")



In [39]:
text="""
LangChain is a framework used to build applications powered by Large Language Models (LLMs). It provides document loaders, text splitters, retrievers, and prompt templates for creating RAG applications.
LangChain supports integration with different LLM providers and vector databases. Developers can use it to retrieve relevant documents and provide contextual information to language models.
Python is a popular, high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data analysis, and artificial intelligence.
Python supports object-oriented, procedural, and functional programming paradigms. Popular libraries such as Pandas, NumPy, and Scikit-learn help developers process data and build machine learning models.
The Taj Mahal is a famous historical monument located in Agra, India. Built by Mughal emperor Shah Jahan in memory of his wife Mumtaz Mahal, it is renowned for its white marble architecture.
Photosynthesis is the process by which green plants use sunlight, water, and carbon dioxide to produce glucose and release oxygen. This process is essential for sustaining life and maintaining oxygen levels in Earth's atmosphere.
"""

In [40]:
class custom_semantic_chunk:
    def __init__(self, model_name="all-MiniLM-L6-v2", threshold=0.5):
        self.threshold = threshold
        self.model = SentenceTransformer(model_name)

    def split(self, text_v):
        sentences = [s.strip() for s in text_v.split("\n") if s.strip()]
        if not sentences:
            return []

        embedding = self.model.encode(sentences)
        chunks=[]
        current_chunk=[sentences[0]]

        for i in range(1, len(sentences)):
            simi = cosine_similarity([embedding[i - 1]], [embedding[i]])[0][0]

            if simi >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]
        chunks.append(" ".join(current_chunk))
        return chunks

    def split_document(self, docs):
        for doc in docs:
            result=[]
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))
        return result




In [41]:
chunker = custom_semantic_chunk()
documents = Document(page_content=text)
result = chunker.split_document([documents])

for idx, chunk in enumerate(result):
    print(f"Chunk: {idx +1} \n: {chunk}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9284.42it/s]


Chunk: 1 
: page_content='LangChain is a framework used to build applications powered by Large Language Models (LLMs). It provides document loaders, text splitters, retrievers, and prompt templates for creating RAG applications. LangChain supports integration with different LLM providers and vector databases. Developers can use it to retrieve relevant documents and provide contextual information to language models.'
Chunk: 2 
: page_content='Python is a popular, high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data analysis, and artificial intelligence. Python supports object-oriented, procedural, and functional programming paradigms. Popular libraries such as Pandas, NumPy, and Scikit-learn help developers process data and build machine learning models.'
Chunk: 3 
: page_content='The Taj Mahal is a famous historical monument located in Agra, India. Built by Mughal emperor Shah Jahan in memory of his wife Mumtaz

In [42]:
vectorstore = FAISS.from_documents(
    documents=result,
    embedding=OpenAIEmbeddings(
        model="text-embedding-3-small",
        dimensions=1536
    )
)

vectorstore

retriever = vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x311e80410>, search_kwargs={})

In [43]:
from langchain_core.prompts import PromptTemplate
template = """Answer the question based on following Context:
 {context}
Question: {question}"""
prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on following Context:\n {context}\nQuestion: {question}')

In [50]:
from langchain_core.runnables import RunnableMap
from langchain_core.output_parsers import StrOutputParser
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

rag_chain = (
    RunnableMap(
        {
            "context":lambda x: retriever.invoke(x["question"]),
            "question": lambda x: x["question"],
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [51]:
resp = rag_chain.invoke({"question": "What is python ?"})
resp


'Python is a popular, high-level programming language known for its simple and readable syntax. It is widely used in various domains including web development, automation, data analysis, and artificial intelligence. Python supports multiple programming paradigms such as object-oriented, procedural, and functional programming. Additionally, it has a rich ecosystem of libraries, such as Pandas, NumPy, and Scikit-learn, which assist developers in processing data and building machine learning models.'